In [83]:
import math
import os
import sys
import getpass
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import random
import string
import json
from datetime import datetime, timedelta
import copy

import typing

device = "cuda" if torch.cuda.is_available() else "cpu"

def is_notebook():
    return 'ipykernel' in sys.modules

print("Python Version: {}".format(sys.version))
print("PyTorch Version: {}".format(torch.__version__))

Python Version: 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
PyTorch Version: 2.10.0+cu126


In [84]:
KEY_POOL = [
    "id", "user_id", "uuid", "session_id", "trace_id", "request_id",
    "name", "username", "first_name", "last_name", "full_name",
    "email", "phone", "password_hash", "auth_token",
    "profile", "account", "settings", "preferences", "metadata",
    "status", "role", "permissions", "group", "tier", "plan",
    "address", "street", "city", "state", "country", "postcode",
    "latitude", "longitude", "geo", "location",
    "timestamp", "created_at", "updated_at", "deleted_at",
    "start_time", "end_time", "duration", "event_time",
    "log_level", "message", "event", "event_type",
    "request", "response", "payload", "headers", "body",
    "endpoint", "route", "method", "query", "params",
    "status_code", "error", "error_code", "error_message",
    "data", "value", "values", "count", "total", "sum",
    "average", "min", "max", "score", "metric", "metrics",
    "features", "embedding", "vector",
    "items", "list", "array", "results", "records",
    "entries", "children", "nodes", "edges",
    "config", "configuration", "version", "build", "environment",
    "service", "service_name", "host", "ip", "port",
    "region", "cluster", "node", "instance",
    "product", "product_id", "price", "currency", "quantity",
    "order", "order_id", "cart", "invoice", "payment",
    "transaction", "amount", "discount", "tax",
    "enabled", "disabled", "active", "inactive",
    "success", "failure", "pending", "retry",
    "flag", "is_valid", "is_deleted",
    "model", "prediction", "label", "confidence",
    "input", "output", "target", "loss",
    "embedding_dim", "attention", "layer",
    "debug", "trace", "stacktrace", "warning",
    "notes", "comment", "description", "tags",
    "category", "type", "kind", "source", "origin"
]

VALUE_GENERATORS = [
    lambda: random.randint(0, 10000),
    lambda: round(random.random() * 1000, random.randint(1, 5)),
    lambda: "".join(random.choices(string.ascii_letters, k=random.randint(3, 12))),
    lambda: random.choice([True, False]),
    lambda: None
]

def generate_value(depth, max_depth):
    """Generate nested JSON values with controlled complexity."""

    # Stop recursion
    if depth >= max_depth:
        return random.choice(VALUE_GENERATORS)()

    choice = random.random()

    # Primitive value
    if choice < 0.4:
        return random.choice(VALUE_GENERATORS)()

    # Nested object
    elif choice < 0.75:
        return generate_object(depth + 1, max_depth)

    # Array
    else:
        return generate_array(depth + 1, max_depth)


def generate_object(depth=0, max_depth=4, max_keys=6):
    """Generate a random JSON object."""

    obj = {}
    num_keys = random.randint(1, max_keys)
    randCount = 0
    
    for item in range(max_keys):
        randCount += 1
        if random.randint(1,2) == 1:
            break

    num_keys = randCount

    for _ in range(num_keys):
        key = random.choice(KEY_POOL)

        # Ensure some uniqueness pressure
        if random.random() < 0.3:
            key += "_" + "".join(random.choices(string.ascii_lowercase, k=3))

        obj[key] = generate_value(depth, max_depth)

    # Occasionally inject missing fields
    if random.random() < 0.2 and len(obj) > 1:
        obj.pop(random.choice(list(obj.keys())))

    return obj


def generate_array(depth=0, max_depth=4, max_len=8):
    """Generate a JSON array with mixed structures."""

    length = random.randint(1, max_len)
    arr = []

    for _ in range(length):
        arr.append(generate_value(depth, max_depth))

    return arr


def generate_complex_json(num_samples=1, max_depth=5, max_keys=6):
    """Generate a list of complex JSON objects."""

    samples = []

    for _ in range(num_samples):
        
        obj = generate_object(0, max_depth, 5)
        
        samples.append(obj)

    return samples

In [85]:
class WeightedJsonCorruptor:
    def __init__(self, weights=None):
        # default weights (tunable)
        self.weights = weights or {
            "struct": 0.25,   # { } [ ] : , "
            "digit": 0.10,
            "alpha": 0.08,
            "space": 0.05,
            "other": 0.15,
            "delete": 0.10,
            "duplicate": 0.08,
            "swap": 0.05,
            "insert_noise": 0.10
        }

        self.struct_chars = set('{}[]":,')
        self.noise_chars = string.ascii_letters + string.digits + "[]{}:,\n\t\""


    def char_type(self, c):
        if c in self.struct_chars:
            return "struct"
        elif c.isdigit():
            return "digit"
        elif c.isalpha():
            return "alpha"
        elif c.isspace():
            return "space"
        else:
            return "other"


    def corrupt(self, text: str, times: int = 1) -> str:
        time = random.randint(1, times)
        chars = list(text)

        randCount = 1
    
        for item in range(times):
            randCount += 1
            if random.randint(1,2) == 1:
                break

        times = randCount

        for num in range(times):
            i = random.randint(1, len(chars)-1)

            c = chars[i]
            ctype = self.char_type(c)

            # 1. DELETE
            if random.random() < self.weights["delete"]:
                chars.pop(i)
                continue

            # 2. DUPLICATE
            elif random.random() < self.weights["duplicate"]:
                chars.insert(i, c)
                i += 1

            # 3. SWAP (local jitter)
            elif i < len(chars) - 1 and random.random() < self.weights["swap"]:
                chars[i], chars[i + 1] = chars[i + 1], chars[i]

            # 4. STRUCTURAL corruption (VERY important for JSON repair)
            elif random.random() < self.weights["struct"]:

                matching_indices = [i for i, c in enumerate(chars) if c in self.struct_chars]
                if matching_indices:
                    i = random.choice(matching_indices)

                if random.random() < 0.9:
                    chars.pop(i)
                else:
                    chars[i] = random.choice(self.noise_chars)

            # 5. DIGIT corruption
            elif ctype == "digit" and random.random() < self.weights["digit"]:
                chars[i] = random.choice(string.digits)

            # 6. ALPHA corruption
            elif ctype == "alpha" and random.random() < self.weights["alpha"]:
                chars[i] = random.choice(string.ascii_letters)

            # 7. OTHER corruption
            elif ctype == "other" and random.random() < self.weights["other"]:
                chars[i] = random.choice(self.noise_chars)

            # 8. INSERT noise before/after
            elif random.random() < self.weights["insert_noise"]:
                noise = random.choice(self.noise_chars)
                if random.random() < 0.5:
                    chars.insert(i, noise)
                    i += 1
                else:
                    chars.insert(i + 1, noise)


        return "".join(chars)
    
corruptor = WeightedJsonCorruptor(
    weights={
        "struct": 1.0,
        "digit": 0.0,
        "alpha": 0.0,
        "space": 0.0,
        "other": 0.0,
        "delete": 0.0,
        "duplicate": 0.0,
        "swap": 0.0,
        "insert_noise": 0.0
    }
)

In [86]:
class CharTokenizer:
    def __init__(self, text: str):
        chars = sorted(set(text))

        self.special = ["<pad>", "<bos>", "<eos>", "<unk>"]
        chars = self.special + chars

        self.stoi = {c: i for i, c in enumerate(chars)}
        self.itos = {i: c for c, i in self.stoi.items()}

        self.vocab_size = len(chars)

        self.pad_id = self.stoi["<pad>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]
        self.unk_id = self.stoi["<unk>"]

    def encode(self, text: str) -> list[int]:
        return [
            self.stoi.get(c, self.unk_id)
            for c in text
        ]

    def decode(self, ids: list[int]) -> str:
        return ''.join(
            self.itos[i]
            for i in ids
            if i in self.itos and self.itos[i] not in self.special
        )

In [87]:
text = "hello world"
vocab = ""
for i in range(32, 127): 
    vocab += (chr(i))

tokenizer = CharTokenizer(vocab)
encoded = tokenizer.encode(text)
decoded = tokenizer.decode(encoded)
print("Original text:", text)
print("Encoded:", encoded)
print("Decoded:", decoded)

Original text: hello world
Encoded: [76, 73, 80, 80, 83, 4, 91, 83, 86, 80, 72]
Decoded: hello world


In [88]:
class JsonDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len: int):
        self.pairs = pairs
        self.tok = tokenizer
        self.max_len = max_len

    def encode_sequence(self, text: str):
        ids = [self.tok.bos_id] + self.tok.encode(text) + [self.tok.eos_id]
        
        if len(ids) > self.max_len:
            ids = ids[:self.max_len]
            
        return torch.tensor(ids, dtype=torch.long)

    def __len__(self): 
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return self.encode_sequence(src), self.encode_sequence(tgt)

In [89]:
def collate_fn(batch):
    src, tgt = zip(*batch)
    return torch.stack(src), torch.stack(tgt)

In [90]:
def dynamic_collate_fn(batch, pad_id=0):
    """
    batch is a list of tuples: [(src_tensor_1, tgt_tensor_1), (src_tensor_2, tgt_tensor_2), ...]
    """
    src_list, tgt_list = zip(*batch)
    
    src_padded = pad_sequence(src_list, batch_first=True, padding_value=pad_id)
    tgt_padded = pad_sequence(tgt_list, batch_first=True, padding_value=pad_id)
    
    return src_padded, tgt_padded

In [91]:
class WordEmbedding(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, max_context_len: int):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb   = nn.Embedding(max_context_len, embed_dim)

    def forward(self, idx: torch.Tensor, start_pos: int = 0) -> torch.Tensor:
        B, T = idx.shape
        pos = torch.arange(start_pos, start_pos + T, device=idx.device)
        return self.token_emb(idx) + self.pos_emb(pos)

In [92]:
class MaskedMultiHeadSelfAttn(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, max_context_len: int):
        super().__init__()

        self.embed_dim   = embed_dim
        self.num_heads   = num_heads
        self.head_dim    = embed_dim // num_heads

        self.q_proj    = nn.Linear(embed_dim, embed_dim)
        self.k_proj    = nn.Linear(embed_dim, embed_dim)
        self.v_proj    = nn.Linear(embed_dim, embed_dim)
        self.out_proj  = nn.Linear(embed_dim, embed_dim)

        mask = torch.tril(torch.ones(max_context_len, max_context_len)).bool()
        self.register_buffer("mask", mask.view(1, 1, max_context_len, max_context_len))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        attn_scores = attn_scores.masked_fill(~self.mask[:, :, :T, :T], float('-inf'))

        attn_weights = torch.softmax(attn_scores, dim=-1)
        out = attn_weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(out)

In [93]:
class MultiHeadCrossAttn(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: torch.Tensor, enc_out: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        _, S, _ = enc_out.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(enc_out).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(enc_out).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        out = torch.softmax(attn_scores, dim=-1) @ v
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

In [94]:
class MultiHeadSelfAttn(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, max_context_len: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        out = torch.softmax(attn_scores, dim=-1) @ v
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

In [95]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, max_context_len: int):
        super().__init__()
        self.ln1  = nn.LayerNorm(embed_dim)
        self.attn = MaskedMultiHeadSelfAttn(embed_dim, num_heads, max_context_len)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.cross_attn = MultiHeadCrossAttn(embed_dim, num_heads)
        self.ln3 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim), 
            nn.ReLU(), 
            nn.Linear(ff_dim, embed_dim)
            )

    def forward(self, x: torch.Tensor, enc_out: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), enc_out)
        x = x + self.ff(self.ln3(x))
        return x

In [96]:
class EncoderBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, max_context_len: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttn(embed_dim, num_heads, max_context_len)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim), 
            nn.ReLU(), 
            nn.Linear(ff_dim, embed_dim)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [114]:
CONTEXT_LEN  = 1024     # sequence length (tokens)
EMBED_DIM    = 128      # model width
NUM_HEADS    = 2        # attention heads  (must divide EMBED_DIM)
NUM_LAYERS   = 2        # transformer blocks
FF_DIM       = 256      # feed-forward hidden size

BATCH_SIZE   = 16
EPOCHS       = 3
LR           = 1e-3 #3e-4
EVAL_SPLIT   = 0.1      # fraction of tokens held out for validation
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

class JsonTransformer(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = EMBED_DIM, num_heads: int = NUM_HEADS, num_layers: int = NUM_LAYERS, ff_dim: int = FF_DIM, max_context_len: int = CONTEXT_LEN):
        super().__init__()
        self.embedding = WordEmbedding(vocab_size, embed_dim, max_context_len)
        self.decoder = nn.Sequential(*[DecoderBlock(embed_dim, num_heads, ff_dim, max_context_len) for _ in range(num_layers)])
        self.encoder = nn.Sequential(*[EncoderBlock(embed_dim, num_heads, ff_dim, max_context_len) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.head.weight = self.embedding.token_emb.weight
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, std=0.02)
                if module.bias is not None: nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=0.02)

    def forward(self, sources: torch.Tensor, targets: torch.Tensor = None):
        enc = self.embedding(sources, start_pos=0)
        for layer in self.encoder:
            enc = layer(enc)

        loss = None
        if targets is not None:
            dec_input = targets[:, :-1]
            dec_targets = targets[:, 1:]
            
            # Positions match the targets slice perfectly starting at 0
            dec = self.embedding(dec_input, start_pos=0) 
            for layer in self.decoder:
                dec = layer(dec, enc)
            dec = self.ln_f(dec)
            logits = self.head(dec)

            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, V), 
                dec_targets.reshape(B * T), 
                ignore_index=0 
            )
        else:
            dec = self.embedding(sources, start_pos=0)
            for layer in self.decoder:
                dec = layer(dec, enc)
            logits = self.head(self.ln_f(dec))

        return logits, loss
    
    def generate(self, source: torch.Tensor, max_new_tokens: int, bos_id: int, eos_id: int):
        self.eval()
        device = source.device
        B = source.size(0)

        enc = self.embedding(source, start_pos=0)
        for layer in self.encoder:
            enc = layer(enc)

        dec_tokens = torch.full((B, 1), bos_id, dtype=torch.long, device=device)

        for i in range(max_new_tokens):
            # Scale step-by-step: pass the full running sequence, but positions map to sequence length
            dec = self.embedding(dec_tokens, start_pos=0)
            x = dec
            for layer in self.decoder:
                x = layer(x, enc)

            logits = self.head(self.ln_f(x[:, -1, :]))
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
            
            dec_tokens = torch.cat([dec_tokens, next_token], dim=1)

            if next_token.item() == eos_id:
                break

        return dec_tokens

In [98]:
def train(model, loader: DataLoader, optimizer: torch.optim.Optimizer) -> float:
    model.train()
    total_loss = 0.0
    for src, tgt in loader:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)
        _, loss = model(src, tgt)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)
@torch.no_grad()
def evaluate(model, loader: DataLoader) -> float:
    model.eval()
    total_loss = 0.0
    
    for src, tgt in loader:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)
       
        _, loss = model(src, tgt)
        
        total_loss += loss.item()
        
    return total_loss / len(loader)

In [99]:
stable_vocab = string.printable 
tokenizer = CharTokenizer(stable_vocab)

In [100]:
model = JsonTransformer(
    vocab_size=tokenizer.vocab_size,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim=FF_DIM,
    max_context_len=CONTEXT_LEN
).to(DEVICE)

In [101]:
#model.load_state_dict(torch.load("transformer.pth"))

In [123]:
examples = generate_complex_json(num_samples=2**16, max_depth=3, max_keys=4)
data = [json.dumps(item) for item in examples]
dataSet = [(corruptor.corrupt(item, 2), item) for item in data]
validationExamples = generate_complex_json(num_samples=2**10, max_depth=3, max_keys=4)
validationData = [json.dumps(item) for item in validationExamples]
validationDataSet = [(corruptor.corrupt(item, 2), item) for item in validationData]

In [124]:
print(f"Vocabulary size : {tokenizer.vocab_size}")

dataset = JsonDataset(dataSet, tokenizer, max_len=CONTEXT_LEN)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: dynamic_collate_fn(b, pad_id=tokenizer.pad_id))

validationDataset = JsonDataset(validationDataSet, tokenizer, max_len=CONTEXT_LEN)
validationLoader = DataLoader(validationDataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: dynamic_collate_fn(b, pad_id=tokenizer.pad_id))

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters      : {n_params:,}\n")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR / 10)

for epoch in range(1, EPOCHS + 1):
    train_loss = train(model, loader, optimizer)
    val_loss = evaluate(model, validationLoader)
    scheduler.step()
    print(f"Epoch {epoch:>3}/{EPOCHS}"f" train={train_loss:.4f} val={val_loss:.4f}"f" ppl={math.exp(val_loss):.1f}")

Vocabulary size : 104
Parameters      : 807,168

Epoch   1/3 train=0.4095 val=0.3308 ppl=1.4


KeyboardInterrupt: 

In [125]:
torch.save(model.state_dict(), "transformerWFixes.pth")

In [160]:
original_text = json.dumps(generate_complex_json(1, max_depth=1, max_keys=4)[0])
#original_text = '{"name": "john"}'
seed_text = corruptor.corrupt(original_text, 2)
seed_ids = torch.tensor(tokenizer.encode(seed_text), dtype=torch.long, device=DEVICE).unsqueeze(0)
out_ids = model.generate(seed_ids, max_new_tokens=CONTEXT_LEN, bos_id=tokenizer.bos_id, eos_id=tokenizer.eos_id)
out_text = tokenizer.decode(out_ids[0].tolist())
print(seed_text)
print(out_text)
print(original_text)


{"deleted_at": {"order_id_xsm" 7063, "username": 288.7}, "flag": true, "created_at": "qCpnovyq", "edges": {"session_id": false, "inactive" 392.22, "payment_aek": 5336}
{"deleted_at": {"order_id_xsm": 7063, "username": 28.87}, "flag": true, "created_at": "qCpnovyq", "edges": {"session": false, "inve": 223.2, "payment_eka": 3633}}}
{"deleted_at": {"order_id_xsm": 7063, "username": 288.7}, "flag": true, "created_at": "qCpnovyq", "edges": {"session_id": false, "inactive": 392.22}, "payment_aek": 5336}
